# Many false constraints at scale: 20 variables, 50 constraints, 5% false

The single-variable [robust_false_constraint](robust_false_constraint.ipynb) notebook
studied *one* bad constraint among four. This scales the same question up: **20
continuous variables** on `[0, 100]`, **50 stated constraints** (means and threshold
probabilities), of which **5% (3 constraints) are completely false** — targets that
flatly contradict the trusted ones on the same variable.

The ground truth is a set of 20 discretised-Gaussian marginals we *choose*; every
true constraint is read off that truth, so the 47 true ones are mutually consistent
and exactly representable. The 3 false constraints are each planted on a variable
that **keeps its true mean as an anchor**, so the conflict is real (1-against-rest
within a variable), not just an unfalsifiable wrong number.

We compare two fits:

| fit | how the 50 constraints enter |
|---|---|
| **ordinary (hard)** | every constraint a hard squared-error term — no notion of trust |
| **on/off robust** | every constraint an `onoff_expectation`: a 2-state trust latent gating a *marginal* penalty |

Why `onoff_expectation` and not the spike-slab / belief latents of the small
notebook? It is the only one that **scales**: it couples the marginal `E[f(X)]`
(gated by the latent's own 2-state marginal), so the latent need not sit adjacent
to its variable and costs **2 bins** instead of ~25. With 50 constraints that is a
70-site chain of mostly-tiny latents rather than an intractable one. It was also
the only no-flag method that cleanly arbitrated in the small study (§2e there).

> **This notebook is set up but not executed** — run it top to bottom to check
> convergence. The scale knobs (`NV`, `NC`, `NB`, `STEPS`, `BOND`) are at the top
> of the setup cell; shrink them for a quick smoke test, grow `STEPS` if the
> on/off gate has not fully converted (trust on the false ones not yet ~0).

## 0. Setup — ground truth, 50 constraints, and the 3 planted lies

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import numpy as np, jax.numpy as jnp
import matplotlib.pyplot as plt
from calibrated_response.tn import TensorChain, ContinuousVar, latent_var, losses as L

# ---- scale knobs (shrink for a smoke test) ----
NV, NC, NB = 20, 50, 20        # variables, constraints, bins per data variable
STEPS, BOND, LR = 3000, 6, 3e-2
N_FALSE = max(1, round(0.05 * NC))   # 5% -> 3
LO, HI = 0.0, 100.0
SEED = 0
rng = np.random.default_rng(SEED)
ind = lambda t: (lambda x: (np.asarray(x) > t).astype(np.float64))   # indicator 1{x>t}

# ---- ground-truth per-variable marginals: discretised gaussians on the bin grid ----
edges = np.linspace(LO, HI, NB + 1); cen = 0.5 * (edges[:-1] + edges[1:])
means = rng.uniform(30, 70, NV); sds = rng.uniform(10, 20, NV)
pmf = [ (lambda p: p / p.sum())(np.exp(-0.5 * ((cen - means[i]) / sds[i]) ** 2))
        for i in range(NV) ]
true_mean = np.array([float((cen * pmf[i]).sum()) for i in range(NV)])
true_pgt  = lambda i, t: float(pmf[i][cen > t].sum())

# ---- 50 constraints: one mean per variable (20) + (NC-NV) threshold probabilities ----
# each constraint: [kind, var, target, value_sd, threshold]
cons = [['mean', i, true_mean[i], 1.0, None] for i in range(NV)]
THR = [30., 50., 70.]
for _ in range(NC - NV):
    i = int(rng.integers(NV)); t = float(rng.choice(THR))
    cons.append(['prob', i, true_pgt(i, t), 0.02, t])

# ---- plant N_FALSE lies: each on a variable that KEEPS its true mean anchor ----
# (falsify a probability constraint; flip its target to the opposite extreme)
prob_by_var = {}
for k, c in enumerate(cons):
    if c[0] == 'prob': prob_by_var.setdefault(c[1], []).append(k)
false_vars = rng.choice(list(prob_by_var), size=N_FALSE, replace=False)
false_idx = []
for v in false_vars:
    k = int(rng.choice(prob_by_var[v])); false_idx.append(k)
    true_p = cons[k][2]
    cons[k][2] = 0.05 if true_p > 0.5 else 0.95      # a flat contradiction of the truth
false_idx = set(false_idx)

def describe(k):
    kind, i, tgt, sd, t = cons[k]
    s = f'E[X{i}]={tgt:5.1f}' if kind == 'mean' else f'P(X{i}>{t:.0f})={tgt:.2f}'
    return s

print(f'{NV} variables, {len(cons)} constraints, {len(false_idx)} planted false ({100*len(false_idx)/len(cons):.0f}%)')
print('planted false constraints (var keeps its true mean anchor):')
for k in sorted(false_idx):
    i = cons[k][1]; t = cons[k][4]
    print(f'  #{k:2d}  {describe(k):16s}   (truth P(X{i}>{t:.0f}) = {true_pgt(i, t):.2f}, true E[X{i}] = {true_mean[i]:.1f})')

## 1. Ordinary fit — all 50 constraints hard

Every constraint is a hard squared-error term with no notion of trust (mean
residuals down-weighted to `~1e-3` so they share a scale with the `[0,1]`
probability residuals, per the small notebook's convention). The 3 false
probabilities cannot be reconciled with their variables' true means, so the
optimiser splits the difference and corrupts exactly those variables — with
nothing in the output flagging which.

In [ ]:
vars_data   = [ContinuousVar(f'X{i}', LO, HI, NB) for i in range(NV)]
vars_latent = [latent_var(f't{k}', 0., 1., 2) for k in range(NC)]   # 2-state trust latents

def recovered_means(m, p):
    return np.array([float(m.expectation(p, {i: cen})) for i in range(NV)])

# ordinary: data variables only need to carry the fit; no latents required
mN = TensorChain(vars_data, bond_dim=BOND, kind='born')
hard = []
for kind, i, tgt, sd, t in cons:
    if kind == 'mean': hard.append(('expect', {i: cen}, tgt, 3e-3))
    else:              hard.append(('prob', {i: mN.threshold_mask(i, t)}, tgt))
pN, hN = mN.optimize(mN.constraint_loss(hard), backend='adam', steps=STEPS, lr=LR, seed=0)

recN = recovered_means(mN, pN)
errN = np.abs(recN - true_mean)
fvars = sorted({cons[k][1] for k in false_idx})
print(f'ordinary (all 50 hard):   mean|E[X]-truth| = {errN.mean():5.2f}   max = {errN.max():5.2f}')
print(f'  on the {len(fvars)} corrupted variables {fvars}: err = {np.round(errN[fvars],1)}')

## 2. On/off robust fit — every constraint gated by a 2-state trust latent

Same 50 statements, but each enters as an `onoff_expectation`: a latent `t_k`
with state 0 = *broken* (asserts nothing) and state 1 = *active*, penalising
`(q_active · (E[f(X)] − target))²/(2 sd²)` plus `KL(q ‖ [p_broken, 1−p_broken])`.
The gate is division-free, so convicting a constraint (`q_active → 0`) costs
exactly `−log p_broken ≈ 3` nats and switches its pull off. The 3 false
constraints conflict with their variables' true means, so their latents flee to
*broken* while the 47 true ones stay near the prior `1 − p_broken = 0.95`.
No per-constraint weight tuning: the belief `(target, value_sd, p_broken)`
carries the scale.

In [ ]:
# on/off: 20 data variables followed by 50 two-state trust latents (70-site chain)
mO = TensorChain(vars_data + vars_latent, bond_dim=BOND, kind='born')
regsO = []
for k, (kind, i, tgt, sd, t) in enumerate(cons):
    f = None if kind == 'mean' else ind(t)
    regsO.append((L.onoff_expectation(i, NV + k, tgt, sd, f=f, p_broken=0.05), 1.0))
pO, hO = mO.optimize(L.combined_loss(mO, [], regsO), backend='adam', steps=STEPS, lr=LR, seed=0)

recO = recovered_means(mO, pO)
errO = np.abs(recO - true_mean)
trust = np.array([float(np.asarray(mO.joint_marginal(pO, NV + k))[1]) for k in range(NC)])
true_k = [k for k in range(NC) if k not in false_idx]

print(f'on/off robust (all 50):   mean|E[X]-truth| = {errO.mean():5.2f}   max = {errO.max():5.2f}')
print(f'  on the {len(fvars)} corrupted variables {fvars}: err = {np.round(errO[fvars],1)}')
print(f'  trust q(active) on the {len(false_idx)} FALSE : {np.round([trust[k] for k in sorted(false_idx)], 2)}   (want ~0)')
print(f'  trust q(active) on the {len(true_k)} TRUE  : min {min(trust[k] for k in true_k):.2f}  mean {np.mean([trust[k] for k in true_k]):.2f}   (want ~0.95)')
print(f'  final loss {float(hO[-1]):.2f} nats vs {len(false_idx)}·(-log 0.05) = {len(false_idx)*-np.log(0.05):.2f}  (floor = prior cost of convicting the false ones)')

## 3. The picture — recovery and trust, side by side

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# left: per-variable mean-recovery error, ordinary vs on/off
x = np.arange(NV); w = 0.4
ax[0].bar(x - w/2, errN, w, color='crimson',   label='ordinary (all hard)')
ax[0].bar(x + w/2, errO, w, color='seagreen',   label='on/off robust')
for v in fvars:
    ax[0].axvspan(v - 0.5, v + 0.5, color='crimson', alpha=0.07)
ax[0].set_xlabel('variable index'); ax[0].set_ylabel('|E[X] - truth|')
ax[0].set_title('mean recovery per variable (shaded = has a false constraint)')
ax[0].set_xticks(x); ax[0].legend(fontsize=9)

# right: trust per constraint (on/off), false ones highlighted
order = list(range(NC))
colors = ['crimson' if k in false_idx else 'seagreen' for k in order]
ax[1].bar(range(NC), trust, color=colors)
ax[1].axhline(0.95, color='0.5', lw=1, ls=':')
ax[1].text(NC - 0.5, 0.955, 'prior trust 0.95', fontsize=8, color='0.4', va='bottom', ha='right')
ax[1].set_xlabel('constraint index'); ax[1].set_ylabel('posterior trust q(active)')
ax[1].set_title('who do you believe? (red = planted false)'); ax[1].set_ylim(0, 1.05)
fig.tight_layout()
_FIG = fig

## What to take away

- **Ordinary hard fitting has no notion of trust.** The 3 false probabilities
  drag their variables' means well off the truth, and the output gives no hint
  which variables were corrupted — the error is spread silently.
- **The on/off robust fit self-arbitrates at scale.** Each false constraint's
  trust latent converts to *broken* (`q(active) → 0`) because it cannot be
  reconciled with its variable's trusted mean, while the 47 true constraints sit
  at the prior `0.95`. The recovered means land back on the ground truth, and the
  trust bar chart *names the culprits* — no flag was supplied.
- **It is the method that scales.** 2-state latents (not 25-bin value grids) and
  a marginal coupling (so latents need not be chain-adjacent to their variables)
  make 50 robust constraints a routine 70-site fit. All beliefs live in
  `(target, value_sd, p_broken)`; there are no per-constraint stiffness knobs.

Honest limits carry over from the small study: a perfectly symmetric conflict is
convicted arbitrarily rather than hedged, and the trust latent carries no scenario
structure — only its marginal `q` is meaningful. Everything here is exact:
`expectation` and the latent `joint_marginal` are contractions of the network, so
both the recovery and the trust readout are defined on the model's true marginals.